#### Data cleaning and aggregation

In [1]:
import numpy as np
import pandas as pd

# Load raw datasets
df_wave = pd.read_csv('M01_wave.csv')
df_wind = pd.read_csv('M01_met.csv')
df_wave['TmStamp'] = pd.to_datetime(df_wave['TmStamp'], format='mixed').dt.tz_localize(None)
df_wind['TmStamp'] = pd.to_datetime(df_wind['TmStamp'], format='mixed').dt.tz_localize(None)
df_wave = df_wave.dropna().reset_index(drop=True)
df_wind = df_wind.dropna().reset_index(drop=True)
df_wave = df_wave.sort_values('TmStamp').reset_index(drop=True)
df_wind = df_wind.sort_values('TmStamp').reset_index(drop=True)

# In total 156,298 hourly timestamps
date_ranges = [
    ('2003-07-09 05:00:00', '2004-02-19 12:00:00'),  # 5408 hourly timestamps
    ('2004-07-08 23:00:00', '2006-04-04 18:00:00'),  # 15236 hourly timestamps
    ('2006-07-18 08:00:00', '2011-11-20 08:00:00'),  # 46825 hourly timestamps
    ('2012-02-04 15:00:00', '2013-02-09 16:00:00'),  # 8906 hourly timestamps
    ('2013-06-23 17:00:00', '2013-12-18 15:00:00'),  # 4271 hourly timestamps
    ('2014-09-18 18:00:00', '2017-02-13 20:00:00'),  # 21099 hourly timestamps
    ('2017-06-28 20:00:00', '2021-01-03 20:00:00'),  # 30841 hourly timestamps
    ('2021-06-12 04:00:00', '2023-02-04 00:00:00'),  # 14445 hourly timestamps
    ('2023-08-04 00:00:00', '2024-08-24 02:00:00')]  # 9267 hourly timestamps

# Filter date ranges
ndf_wave = pd.DataFrame()
ndf_wind = pd.DataFrame()
for start, end in [(pd.to_datetime(start), pd.to_datetime(end[:-5] + '59:59')) for start, end in date_ranges]:
    ndf_wave = pd.concat([ndf_wave, df_wave[(df_wave['TmStamp'] >= start) & (df_wave['TmStamp'] <= end)]])
    ndf_wind = pd.concat([ndf_wind, df_wind[(df_wind['TmStamp'] >= start) & (df_wind['TmStamp'] <= end)]])

# Resample by hour
ndf_wave['TmStamp'] = ndf_wave['TmStamp'].dt.floor('h')
ndf_wind['TmStamp'] = ndf_wind['TmStamp'].dt.floor('h')
ndf_wave = ndf_wave.loc[ndf_wave.groupby('TmStamp')['H'].idxmax().dropna()]
ndf_wind = ndf_wind.loc[ndf_wind.groupby('TmStamp')['WSPD'].idxmax().dropna()]

# Correct values
ndf_wave['H'] = 3.28084 * ndf_wave['H']  # m to ft

# Fill in the missing timestamps
timestamps = []
for start, end in [(pd.to_datetime(start), pd.to_datetime(end)) for start, end in date_ranges]:
    timestamps.extend(pd.date_range(start, end, freq='h'))

# Merge datasets
df_new = pd.merge(ndf_wave, ndf_wind, on='TmStamp', how='outer')
df_new = pd.merge(pd.DataFrame({'TmStamp':timestamps}), df_new, on='TmStamp', how='left')
df_new = df_new.sort_values('TmStamp').reset_index(drop=True)

#### Data imputation

In [2]:
# Wind data imputation
# Count consecutive NaN lengths
def get_nan_lengths(series):
    nans = series.isna()
    group = (nans != nans.shift()).cumsum()
    nan_lengths = nans.groupby(group).transform('sum')
    return np.where(nans, nan_lengths, 0).astype(int)

# Case 1: Interpolate when nan_lengths ≤ 5
WSPD = df_new['WSPD'].interpolate(method='linear')
WDIR = df_new['WDIR'].interpolate(method='linear')

n_random = np.random.normal(loc=0, scale=1, size=len(df_new))
sd_WSPD = np.std(df_new['WSPD'].dropna())
WSPD = WSPD + n_random * sd_WSPD

WSPD_nans = get_nan_lengths(df_new['WSPD'])
WDIR_nans = get_nan_lengths(df_new['WDIR'])
WSPD_mask = (WSPD_nans > 0) & (WSPD_nans <= 5)
WDIR_mask = (WDIR_nans > 0) & (WDIR_nans <= 5)

df_new.loc[WSPD_mask, 'WSPD'] = WSPD[WSPD_mask]
df_new.loc[WDIR_mask, 'WDIR'] = WDIR[WDIR_mask]

# Case 2: Sample from known ('WSPD', 'WDIR') when nan_lengths > 5
df_new['season'] = np.where(df_new['TmStamp'].dt.month.isin([11, 12, 1, 2, 3]), 'windy', 'calm')
for season in ['windy','calm']:
    df_season = df_new[df_new['season'] == season]
    mask = (WSPD_nans > 5) & (df_new['season'] == season)
    sampled = df_season[['WSPD','WDIR']].dropna().sample(n=mask.sum(), replace=True)
    df_new.loc[mask, ['WSPD','WDIR']] = sampled.values

In [3]:
# Wave data imputation
# Create lagged WSPD quartile bins
df_new['rWSPD'] = df_new['WSPD'].shift(1)
quartiles = df_new['rWSPD'].quantile([0.25, 0.5, 0.75])
df_new['bin'] = pd.cut(
    df_new['rWSPD'],
    bins = [-np.inf, quartiles[0.25], quartiles[0.5], quartiles[0.75], np.inf],
    labels = ['Q1','Q2','Q3','Q4'])

# Impute NaN with the median wave height for each bin categorized by season
damp = {'windy':0.03, 'calm':0.01}
for season in ['windy','calm']:
    df_season = df_new[df_new['season'] == season]
    for label in ['Q1','Q2','Q3','Q4']:
        mask = (df_new['H'].isna()) & (df_new['bin'] == label) & (df_new['season'] == season)
        obs_median = df_season[df_season['bin'] == label]['H'].median()
        n_random = np.random.normal(loc=0, scale=1, size=mask.sum())
        df_new.loc[mask, 'H'] = obs_median + damp[season] * n_random

#### Processed dataset output

In [4]:
df_new['WSPD'] = df_new['WSPD'].clip(lower=0.25)

# Save the processed dataset
df_new = df_new[['TmStamp','H','WSPD','WDIR']]
df_new = df_new.sort_values('TmStamp').reset_index(drop=True)
df_new.to_csv('M01_data.csv', index=False)

df_new.head()

,TmStamp,H,WSPD,WDIR
0,2003-07-09 05:00:00,2.094076,5.444,255.6667
1,2003-07-09 06:00:00,2.163797,1.632,229.7667
2,2003-07-09 07:00:00,2.302210,3.370,222.5667
3,2003-07-09 08:00:00,2.451478,3.467,260.0667
4,2003-07-09 09:00:00,2.094074,5.225,278.3667
